# FrictionLLM + RLCFrictionLM
### A neural architecture based on physical friction and RLC circuit mechanics

**Core idea:**
Current LLMs are pure resistors — every weight fires for every token.
This architecture maps real physics onto neurons:

| Physics | Component | Effect in the network |
|---|---|---|
| Static friction  | μ_s threshold  | Neuron stays **stuck at 0** unless signal exceeds threshold |
| Kinetic friction | μ_k drag       | Once fired, output = z − sign(z)·μ_k  (energy loss) |
| Inductance (L)   | Inertia        | Resists rapid changes in activation |
| Capacitance (C)  | Charge storage | Accumulates signal across **layers** before firing |
| Resonance        | ω₀ = 1/√(LC)  | Each neuron tunes to a natural frequency |

**What emerges:** sparsity (CPU-friendly inference), hierarchical feature tuning, and physically-principled dynamics.


## Setup

Run from the repo root after `pip install torch tiktoken numpy tqdm`.

In [ ]:
import os, math, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## Verify package

In [ ]:
import sys
sys.path.insert(0, '.')   # repo root

from friction_llm import (
    FrictionConfig, FrictionLM, RLCFrictionLM,
    SharpnessCurriculum, RLCNeuron
)
print('friction_llm loaded OK')


## Prepare data

Only needed once. Points at `data/input.txt`.

In [ ]:
import urllib.request, os

os.makedirs("data", exist_ok=True)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("data/input.txt"):
    urllib.request.urlretrieve(url, "data/input.txt")
    print("Downloaded TinyShakespeare")
else:
    print("Data already downloaded")
print(f"Size: {os.path.getsize('data/input.txt'):,} bytes")


In [ ]:
import tiktoken
import numpy as np

enc = tiktoken.get_encoding("gpt2")
with open("data/input.txt") as f:
    text = f.read()

tokens = np.array(enc.encode_ordinary(text), dtype=np.uint16)
split  = int(0.9 * len(tokens))
tokens[:split].tofile("data/train.bin")
tokens[split:].tofile("data/val.bin")
print(f"Train: {split:,} tokens   Val: {len(tokens)-split:,} tokens")


In [ ]:
class TokenDataset:
    def __init__(self, path, seq_len):
        self.data    = np.memmap(path, dtype=np.uint16, mode="r")
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.data) - self.seq_len - 1)

    def get_batch(self, batch_size, device):
        ix = torch.randint(len(self.data) - self.seq_len - 1, (batch_size,))
        x  = torch.stack([torch.from_numpy(self.data[i:i+self.seq_len].astype(np.int64)) for i in ix])
        y  = torch.stack([torch.from_numpy(self.data[i+1:i+1+self.seq_len].astype(np.int64)) for i in ix])
        return x.to(device), y.to(device)

train_ds = TokenDataset("data/train.bin", seq_len=256)
val_ds   = TokenDataset("data/val.bin",   seq_len=256)
print(f"Batches available: {len(train_ds):,}")


## Training

In [ ]:
from friction_llm import FrictionConfig, FrictionLM, RLCFrictionLM, SharpnessCurriculum

def train_model(model, train_ds, val_ds, max_steps=600, batch_size=16,
                lr=3e-4, log_every=50, label="model"):
    use_amp = device.type == "cuda"
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    # Separate physics params (no weight decay)
    physics, other = [], []
    for n, p in model.named_parameters():
        if any(k in n for k in ("raw_mu","raw_ratio","log_L","log_R","log_C")):
            physics.append(p)
        else:
            other.append(p)
    optimizer = torch.optim.AdamW(
        [{"params": other,   "weight_decay": 0.1},
         {"params": physics, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.95)
    )

    curriculum = SharpnessCurriculum(model, model.config)
    history = {"step": [], "loss": [], "val_loss": [], "sparsity": []}
    best_val = float("inf")
    t0 = time.time()

    for step in range(max_steps + 1):
        # LR warmup
        warmup = 200
        cur_lr = lr * min(step / warmup, 1.0)
        for pg in optimizer.param_groups:
            pg["lr"] = cur_lr

        model.train()
        x, y = train_ds.get_batch(batch_size, device)
        with torch.amp.autocast("cuda", enabled=use_amp):
            _, loss = model(x, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        curriculum.step()

        if step % log_every == 0:
            # Validation
            model.eval()
            with torch.no_grad():
                xv, yv = val_ds.get_batch(batch_size, device)
                _, vloss = model(xv, yv)

            # Sparsity
            sample, _ = val_ds.get_batch(4, device)
            if hasattr(model, "circuit_report"):
                report  = model.circuit_report(sample)
                sparsity = report["overall_sparsity"]
            else:
                report  = model.measure_sparsity(sample)
                sparsity = report["overall"]
            model.train()

            dt = (time.time() - t0) / max(step, 1)
            history["step"].append(step)
            history["loss"].append(loss.item())
            history["val_loss"].append(vloss.item())
            history["sparsity"].append(sparsity)
            print(f"[{label}] step {step:4d} | loss {loss.item():.4f} "
                  f"| val {vloss.item():.4f} | sparsity {sparsity:.1%} "
                  f"| {dt*1000:.0f}ms/step")

    return history


### Train FrictionLM (R only)

In [ ]:
cfg_f = FrictionConfig.small()
cfg_f.max_seq_len = 256
model_f = FrictionLM(cfg_f).to(device)
print(f"FrictionLM: {model_f.param_count()/1e6:.1f}M params")

history_f = train_model(model_f, train_ds, val_ds,
                        max_steps=600, batch_size=16, label="FrictionLM")


### Train RLCFrictionLM (full L + R + C)

In [ ]:
cfg_r = FrictionConfig.small()
cfg_r.max_seq_len = 256
cfg_r.use_rlc = True
model_r = RLCFrictionLM(cfg_r).to(device)
print(f"RLCFrictionLM: {model_r.param_count()/1e6:.1f}M params")

history_r = train_model(model_r, train_ds, val_ds,
                        max_steps=600, batch_size=16, label="RLCFrictionLM")


## Circuit physics report

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
sample_text = "To be or not to be, that is the question"
sample_ids  = torch.tensor(enc.encode_ordinary(sample_text),
                           dtype=torch.long, device=device).unsqueeze(0)

model_r.eval()
model_r.print_circuit_report(sample_ids)


## Sparsity analysis

In [ ]:
# Per-layer sparsity breakdown for both models
model_f.eval(); model_r.eval()
sample, _ = val_ds.get_batch(8, device)

print("FrictionLM — per-layer sparsity:")
stats_f = model_f.measure_sparsity(sample)
for k, v in stats_f.items():
    if k == "overall":
        print(f"  OVERALL: {v:.1%}")
    else:
        print(f"  {k}: {v['sparsity']:.1%}  μ_s={v['mu_s']:.3f}  μ_k={v['mu_k']:.3f}")

print()
print("RLCFrictionLM — per-layer circuit stats:")
model_r.print_circuit_report(sample)


## Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 10))
fig.suptitle("FrictionLLM vs RLCFrictionLM — Training Dynamics", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Loss curves ───────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(history_f["step"], history_f["val_loss"],  label="FrictionLM val",  color="#e05a5a", linewidth=2)
ax1.plot(history_r["step"], history_r["val_loss"],  label="RLCFrictionLM val", color="#5a9ce0", linewidth=2)
ax1.plot(history_f["step"], history_f["loss"],  linestyle="--", alpha=0.4, color="#e05a5a")
ax1.plot(history_r["step"], history_r["loss"],  linestyle="--", alpha=0.4, color="#5a9ce0")
ax1.set_xlabel("Step"); ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training & Validation Loss"); ax1.legend(); ax1.grid(alpha=0.3)

# ── Sparsity ──────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(history_f["step"], [s*100 for s in history_f["sparsity"]], color="#e05a5a", linewidth=2, label="FrictionLM")
ax2.plot(history_r["step"], [s*100 for s in history_r["sparsity"]], color="#5a9ce0", linewidth=2, label="RLCFrictionLM")
ax2.axhline(70, linestyle="--", color="gray", alpha=0.5, label="CPU-efficient threshold")
ax2.set_xlabel("Step"); ax2.set_ylabel("Sparsity (%)")
ax2.set_title("Gate Sparsity (higher = more neurons silent)"); ax2.legend(); ax2.grid(alpha=0.3)

# ── RLC natural frequencies per layer ────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
model_r.eval()
omega_per_layer, zeta_per_layer = [], []
for i, block in enumerate(model_r.blocks):
    rlc = block.rlc_block.rlc
    omega_per_layer.append(rlc.omega_0.detach().cpu().numpy())
    zeta_per_layer.append(rlc.damping_ratio.detach().cpu().numpy())

colors = plt.cm.viridis([i / len(omega_per_layer) for i in range(len(omega_per_layer))])
for i, (w, c) in enumerate(zip(omega_per_layer, colors)):
    ax3.hist(w, bins=30, alpha=0.6, color=c, label=f"Layer {i}")
ax3.set_xlabel("Natural Frequency ω₀"); ax3.set_ylabel("Count")
ax3.set_title("ω₀ Distribution per Layer\n(divergence = different frequency bands)"); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# ── Damping ratio per layer ───────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
for i, (z, c) in enumerate(zip(zeta_per_layer, colors)):
    ax4.hist(z, bins=30, alpha=0.6, color=c, label=f"Layer {i}")
ax4.axvline(1.0, color="red", linestyle="--", linewidth=2, label="Critical damping (ζ=1)")
ax4.set_xlabel("Damping Ratio ζ"); ax4.set_ylabel("Count")
ax4.set_title("Damping Ratio per Layer\n(<1=resonant, 1=critical, >1=overdamped)"); ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

# ── μ_s per layer (friction thresholds) ──────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
mu_s_per_layer = []
for block in model_r.blocks:
    mu_s_per_layer.append(block.rlc_block.friction.mu_s.detach().cpu().numpy())
ax5.boxplot(mu_s_per_layer, labels=[f"L{i}" for i in range(len(mu_s_per_layer))])
ax5.set_xlabel("Layer"); ax5.set_ylabel("μ_s value")
ax5.set_title("Static Friction Thresholds μ_s\n(higher = harder to activate)"); ax5.grid(alpha=0.3)

plt.savefig("friction_rlc_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to friction_rlc_analysis.png")


## Text generation

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")

def generate(model, prompt, max_tokens=150, temperature=0.8, top_k=40):
    model.eval()
    ids = enc.encode_ordinary(prompt)
    idx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(idx, max_new_tokens=max_tokens,
                         temperature=temperature, top_k=top_k)
    return enc.decode(out[0].tolist())

prompt = "HAMLET:\nTo be or not to be"

print("=" * 60)
print("FrictionLM output:")
print("=" * 60)
print(generate(model_f, prompt))

print()
print("=" * 60)
print("RLCFrictionLM output:")
print("=" * 60)
print(generate(model_r, prompt))


## Physics concepts demonstrated

| Concept | Where it shows up | How to observe |
|---|---|---|
| Static friction | Sparsity % | Most neurons stay at 0 |
| Kinetic drag | Output amplitude | Active neurons lose μ_k energy |
| Jolt | μ_s − μ_k gap | Neurons jump, don't ramp |
| Capacitance | Charge accumulation | q builds across layers |
| Resonance | ω₀ divergence | Layers evolve different frequencies |
| Damping modes | ζ histogram | Some neurons go underdamped (resonant) |
